# Voice Field Notes Assistant

**Stack:** Whisper (ASR) + Gemini 1.5 Flash (LLM) + gTTS (TTS)

In [45]:
!pip install -q openai-whisper google-generativeai gTTS soundfile sounddevice pydub
print('Done.')

I0430 19:21:50.413592  458366 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(110, generation: 1)


Done.


In [46]:
import os
import io
import time
import base64
import tempfile
import numpy as np
import soundfile as sf
import whisper
import google.generativeai as genai
from gtts import gTTS
from IPython.display import display, Audio

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
if not GEMINI_API_KEY:
    raise ValueError('Set GEMINI_API_KEY environment variable before running.')
genai.configure(api_key=GEMINI_API_KEY)

print('Loading Whisper base model...')
whisper_model = whisper.load_model('base')
print('Whisper model loaded')

gemini_model = genai.GenerativeModel('gemini-2.5-flash')
print('Gemini 1.5 Flash ready')

audio_path = None
transcript_text = None
gemini_response_text = None

Loading Whisper base model...
Whisper model loaded
Gemini 1.5 Flash ready


In [47]:
import sounddevice as sd

RECORD_SECONDS = 10
SAMPLE_RATE = 16000


print(f'Recording {RECORD_SECONDS}s... speak now!')
recording = sd.rec(int(RECORD_SECONDS * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()
print('Recording complete.')

tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
sf.write(tmp.name, recording, SAMPLE_RATE)
audio_path = tmp.name
print(f'Saved to: {audio_path}')

Recording 10s... speak now!
Recording complete.
Saved to: /var/folders/1d/6_kdmv4964vgsqrd5qjtcgxh0000gn/T/tmp748zehdm.wav


In [48]:
if audio_path is None:
    print('No audio found.')
else:
    print(f'Transcribing: {audio_path}')
    result = whisper_model.transcribe(audio_path, fp16=False)
    transcript_text = result['text'].strip()
    detected_lang = result.get('language', 'unknown')

    print('\nTRANSCRIPT')
    print('-' * 40)
    print(transcript_text)
    print(f'Detected language: {detected_lang}')

Transcribing: /var/folders/1d/6_kdmv4964vgsqrd5qjtcgxh0000gn/T/tmp748zehdm.wav


I0430 19:22:01.648505  458570 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(110, generation: 1)



TRANSCRIPT
----------------------------------------
Hello my name is Bplopthava and I am having a headache. What should I do?
Detected language: hi


In [49]:
SYSTEM_PROMPT = """You are a helpful medical assistant.
The user will give you a spoken field observation.
Respond conversationally — answer what they said, clarify key points, and suggest next steps.
Keep it short and natural. Do not use rigid templates or field labels."""

if transcript_text is None:
    print('No transcript found. ')
else:
    prompt = SYSTEM_PROMPT + f'\n\nField note:\n"{transcript_text}"'
    print('Sending to Gemini...')
    response = gemini_model.generate_content(prompt)
    gemini_response_text = response.text.strip()
    print('\n' + gemini_response_text)
    print('\nRun Cell 7 to hear it read back.')

Sending to Gemini...

I'm sorry to hear you're having a headache, Bplopthava. That's no fun at all.

To help me understand a bit better, could you tell me how severe the pain is on a scale of 1 to 10, with 10 being the worst? Also, are you experiencing any other symptoms like nausea, dizziness, sensitivity to light or sound, or any changes in your vision? Knowing a little more about it will help us figure out the best next steps.

Run Cell 7 to hear it read back.


In [50]:
if gemini_response_text is None:
    print('No field note found.')
else:
    tts_text = gemini_response_text.replace(' — ', '. ').replace('—', '. ')
    tts = gTTS(text=tts_text, lang='en', slow=False)
    output_path = './field_note_audio.mp3'
    tts.save(output_path)
    print(f'Audio saved to: {output_path}')
    display(Audio(output_path, autoplay=True))

Audio saved to: ./field_note_audio.mp3
